Script permettant de télécharger openmétéo

In [ ]:
"""Téléchargement des données météo horaires (Open-Meteo, API Archive) pour
12 grandes villes françaises (une par région), de 2021 à 2025.

Usage :
    python telecharger_meteo.py

Résultat :
    data/raw/openmeteo/<ville>_<année>.json   (réponses brutes, servent de cache)
    data/processed/meteo_horaire.parquet      (tableau final, une ligne par ville et par heure)

Dépendances : requests, pandas, pyarrow
Données : Open-Meteo.com (licence CC BY 4.0, usage non commercial) à citer dans le rapport.
"""

import json
import time
from pathlib import Path

import pandas as pd
import requests

URL = "https://archive-api.open-meteo.com/v1/archive"
RAW_DIR = Path("data/raw/openmeteo")
OUT_PATH = Path("data/processed/meteo_horaire.parquet")

ANNEES = range(2021, 2026)  # 2021 à 2025 inclus

HOURLY = [
    "temperature_2m",
    "shortwave_radiation",
    "cloud_cover",
    "wind_speed_10m",
    "wind_speed_100m",
    "precipitation",
]

# Une ville représentative par région (France métropolitaine hors Corse).
VILLES = {
    "paris":      {"lat": 48.8566, "lon": 2.3522,  "region": "Île-de-France"},
    "lyon":       {"lat": 45.7640, "lon": 4.8357,  "region": "Auvergne-Rhône-Alpes"},
    "toulouse":   {"lat": 43.6047, "lon": 1.4442,  "region": "Occitanie"},
    "bordeaux":   {"lat": 44.8378, "lon": -0.5792, "region": "Nouvelle-Aquitaine"},
    "lille":      {"lat": 50.6292, "lon": 3.0573,  "region": "Hauts-de-France"},
    "strasbourg": {"lat": 48.5734, "lon": 7.7521,  "region": "Grand Est"},
    "marseille":  {"lat": 43.2965, "lon": 5.3698,  "region": "Provence-Alpes-Côte d'Azur"},
    "nantes":     {"lat": 47.2184, "lon": -1.5536, "region": "Pays de la Loire"},
    "rennes":     {"lat": 48.1173, "lon": -1.6778, "region": "Bretagne"},
    "rouen":      {"lat": 49.4432, "lon": 1.0993,  "region": "Normandie"},
    "dijon":      {"lat": 47.3220, "lon": 5.0415,  "region": "Bourgogne-Franche-Comté"},
    "orleans":    {"lat": 47.9029, "lon": 1.9093,  "region": "Centre-Val de Loire"},
}


def fetch_year(nom, lat, lon, annee, max_retries=5):
    """Télécharge une année pour une ville. Utilise le cache disque s'il existe."""
    path = RAW_DIR / f"{nom}_{annee}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": f"{annee}-01-01",
        "end_date": f"{annee}-12-31",
        "hourly": ",".join(HOURLY),
        "timezone": "UTC",  # évite les problèmes de changement d'heure
    }

    for tentative in range(max_retries):
        r = requests.get(URL, params=params, timeout=60)
        if r.status_code == 200:
            RAW_DIR.mkdir(parents=True, exist_ok=True)
            path.write_text(r.text, encoding="utf-8")
            return r.json()
        if r.status_code == 429:  # limite d'appels atteinte : on attend puis on réessaie
            attente = 10 * 2**tentative
            print(f"  429 reçu, nouvelle tentative dans {attente}s")
            time.sleep(attente)
            continue
        r.raise_for_status()

    raise RuntimeError(f"Échec du téléchargement pour {nom} {annee}")


def to_dataframe(nom, info, payload):
    df = pd.DataFrame(payload["hourly"])
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df["ville"] = nom
    df["region"] = info["region"]
    return df


def main(pause=1.0):
    frames = []
    for nom, info in VILLES.items():
        for annee in ANNEES:
            print(f"{nom} {annee}")
            payload = fetch_year(nom, info["lat"], info["lon"], annee)
            frames.append(to_dataframe(nom, info, payload))
            time.sleep(pause)  # politesse envers l'API gratuite

    meteo = pd.concat(frames, ignore_index=True)

    # Contrôle : 5 ans d'heures par ville (43 824 lignes), sans doublons
    attendu = len(pd.date_range("2021-01-01", "2025-12-31 23:00", freq="h"))
    comptes = meteo.groupby("ville").size()
    anormales = comptes[comptes != attendu]
    if not anormales.empty:
        print(f"Attention, nombre de lignes inattendu (attendu {attendu}) :\n{anormales}")
    assert not meteo.duplicated(["ville", "time"]).any(), "Doublons détectés"

    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    meteo.to_parquet(OUT_PATH, index=False)
    print(f"{len(meteo)} lignes enregistrées dans {OUT_PATH}")


if __name__ == "__main__":
    main()